In [1]:
import os
import time
from google import genai
from google.genai import types
from dotenv import load_dotenv
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


from datasets import load_dataset

In [2]:
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1066 entries, 0 to 1065
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    1066 non-null   object
 1   label   1066 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 16.8+ KB


In [3]:
test['label'] = test['label'].apply(lambda x: 'positive' if x == 1 else 'negative')

labels = test['label'].unique()

test

,text,label
0,lovingly photographed in the manner of a golde...,positive
1,consistently clever and suspenseful .,positive
2,"it's like a "" big chill "" reunion of the baade...",positive
3,the story gives ample opportunity for large-sc...,positive
4,"red dragon "" never cuts corners .",positive
...,...,...
1061,a terrible movie that some people will neverth...,negative
1062,there are many definitions of 'time waster' bu...,negative
1063,"as it stands , crocodile hunter has the hurrie...",negative
1064,the thing looks like a made-for-home-video qui...,negative


In [4]:
load_dotenv()

api_key=os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

In [5]:
def classify(text, labels):

    sys_instruct="You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions."

    start_time = time.time()

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        config=types.GenerateContentConfig(
            system_instruction=sys_instruct,
            safety_settings=[
            types.SafetySetting(
                category="HARM_CATEGORY_HARASSMENT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_HATE_SPEECH",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_DANGEROUS_CONTENT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_CIVIC_INTEGRITY",
                threshold="BLOCK_NONE"
            ),
            ],
        ),
        contents=f"Classify the following text based on the task: Sentiment analysis of movie reviews. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"
    )

    request_time = time.time() - start_time
    completion = response.text
    if completion:
        completion = completion.lower()
    else:
        completion = "None"
    completion_tokens = response.usage_metadata.candidates_token_count
    prompt_tokens = response.usage_metadata.prompt_token_count
    total_tokens = response.usage_metadata.total_token_count

    return completion, request_time, completion_tokens, prompt_tokens, total_tokens

def post_process(text):
    if 'positive' in text:
        return 'positive'
    elif 'negative' in text:
        return 'negative'
    else:
        return 'error'

In [6]:
pred_df = test.copy() 

for index, row in pred_df.iterrows():
    try:
        text = row['text']
        completion, request_time, completion_tokens, prompt_tokens, total_tokens = classify(text, labels)
        pred_df.at[index, 'prediction'] = completion
        pred_df.at[index, 'request_time'] = request_time
        pred_df.at[index, 'completion_tokens'] = completion_tokens
        pred_df.at[index, 'prompt_tokens'] = prompt_tokens
        pred_df.at[index, 'total_tokens'] = total_tokens

    except Exception as e:
        # Save the current state of the DataFrame to a file before breaking out or retrying.
        pred_df.to_csv("results/partial_gemini_ZS_binary2.csv", index=False)
        print(f"An error occurred at index {index}: {e}. Partial results saved.")
        # Optionally, you can break out of the loop or continue based on your needs.
        break

pred_df['prediction_post_processed'] = pred_df['prediction'].apply(post_process)
pred_df.to_csv("results/gemini_ZS_binary2.csv", index=False)

pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,lovingly photographed in the manner of a golde...,positive,positive\n,1.136801,2.0,100.0,102.0,positive
1,consistently clever and suspenseful .,positive,positive\n,1.035619,2.0,81.0,83.0,positive
2,"it's like a "" big chill "" reunion of the baade...",positive,positive\n,1.111775,2.0,107.0,109.0,positive
3,the story gives ample opportunity for large-sc...,positive,positive\n,1.065924,2.0,99.0,101.0,positive
4,"red dragon "" never cuts corners .",positive,positive\n,1.042670,2.0,82.0,84.0,positive
...,...,...,...,...,...,...,...,...
1061,a terrible movie that some people will neverth...,negative,negative\n,1.051164,2.0,86.0,88.0,negative
1062,there are many definitions of 'time waster' bu...,negative,negative\n,0.503949,2.0,95.0,97.0,negative
1063,"as it stands , crocodile hunter has the hurrie...",negative,negative\n,1.064674,2.0,125.0,127.0,negative
1064,the thing looks like a made-for-home-video qui...,negative,negative\n,0.985738,2.0,90.0,92.0,negative


In [7]:
pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,lovingly photographed in the manner of a golde...,positive,positive\n,1.136801,2.0,100.0,102.0,positive
1,consistently clever and suspenseful .,positive,positive\n,1.035619,2.0,81.0,83.0,positive
2,"it's like a "" big chill "" reunion of the baade...",positive,positive\n,1.111775,2.0,107.0,109.0,positive
3,the story gives ample opportunity for large-sc...,positive,positive\n,1.065924,2.0,99.0,101.0,positive
4,"red dragon "" never cuts corners .",positive,positive\n,1.042670,2.0,82.0,84.0,positive
...,...,...,...,...,...,...,...,...
1061,a terrible movie that some people will neverth...,negative,negative\n,1.051164,2.0,86.0,88.0,negative
1062,there are many definitions of 'time waster' bu...,negative,negative\n,0.503949,2.0,95.0,97.0,negative
1063,"as it stands , crocodile hunter has the hurrie...",negative,negative\n,1.064674,2.0,125.0,127.0,negative
1064,the thing looks like a made-for-home-video qui...,negative,negative\n,0.985738,2.0,90.0,92.0,negative


In [8]:
y_pred = pred_df['prediction_post_processed']
y_true = pred_df['label']

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.928705
F1 score: 0.928675
Precision: 0.929437
Recall: 0.928705


In [9]:
# get average response time, vram usage and ram usage
request_time_avg = pred_df['request_time'].mean()
completion_tokens_avg = pred_df['completion_tokens'].mean()
prompt_tokens_avg = pred_df['prompt_tokens'].mean()
total_tokens_avg = pred_df['total_tokens'].mean()

print(f'Average response time: {request_time_avg}')
print(f'Average completion tokens: {completion_tokens_avg}')
print(f'Average prompt tokens: {prompt_tokens_avg}')
print(f'Average total tokens: {total_tokens_avg}')

Average response time: 0.9638142916767056
Average completion tokens: 2.0
Average prompt tokens: 99.59193245778611
Average total tokens: 101.59193245778611


In [10]:
input_token_price = 0.1/1_000_000
output_token_price = 0.4/1_000_000

# Calculate the cost of the requests
total_cost = 0
for index, row in pred_df.iterrows():
    completion_tokens = row['completion_tokens']
    prompt_tokens = row['prompt_tokens']
    cost  = completion_tokens * output_token_price + prompt_tokens * input_token_price
    total_cost += cost

print(f'Total cost: USD {total_cost}')

Total cost: USD 0.011469300000000017


In [11]:
with open('results/gemini_ZS_binary2.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {request_time_avg}\n')
    f.write(f'Average completion tokens: {completion_tokens_avg}\n')
    f.write(f'Average prompt tokens: {prompt_tokens_avg}\n')
    f.write(f'Average total tokens: {total_tokens_avg}\n')
    f.write(f'Total cost: USD {total_cost}\n')